# 01 — Unified Route Tokenization for TB2 and Kilter

## What is tokenization and why does it matter?

In natural language processing, **tokenization** is the process of converting raw text into a sequence of discrete symbols (tokens) that a model can process. For example, the sentence "I climb rocks" might be tokenized as `["I", " climb", " rocks"]` using a subword tokenizer like BPE.

For climbing board routes, we face an analogous problem: how do we convert a climb — which is fundamentally a *set of holds at specific positions with specific roles* — into a sequence of tokens that a transformer can learn from?

### Key design decisions in this notebook

1. **Board namespacing**: Each hold token includes the board prefix (e.g., `TB2_p344_start` vs `KILTER_p1084_start`). This prevents placement ID collisions between boards — placement 344 on TB2 is a completely different physical hold than placement 344 on Kilter (in fact, the latter does not exist).

2. **Semantic role mapping**: Different boards use different role IDs (TB2 uses 5/6/7/8, Kilter uses 12/13/14/15), but they all map to the same semantic roles: `start`, `middle`, `finish`, `foot`. This shared vocabulary lets the model learn transferable patterns.

3. **Canonical ordering**: Holds within a route are sorted by (role priority, y-position, x-position). This gives the model a consistent input ordering, similar to how LLMs expect text in left-to-right order.

4. **Special tokens**: Like BERT and GPT, we use special tokens:
   - `<BOS>` (beginning of sequence) — marks the start, like `[CLS]` in BERT
   - `<EOS>` (end of sequence) — marks the end, like `[SEP]` or the end-of-text token in GPT
   - `<PAD>` — for batching sequences of different lengths
   - `<UNK>` — for unknown tokens (safety net)
   - `<CLS>` — used by the grade predictor to pool sequence information
   - `<MASK>` — reserved for future masked language modeling experiments

5. **Conditioning tokens**: Routes are prefixed with board, angle, and grade tokens. This is analogous to how modern LLMs use system prompts or task prefixes to condition generation.

### The analogy to NLP

| NLP Concept | Climbing Board Analog |
|---|---|
| Word / Subword | Hold token (placement + role) |
| Sentence | Route (sequence of holds) |
| Document language | Board type (TB2 vs Kilter) |
| Sentence length | Number of holds in route |
| POS tag | Semantic role (start/middle/finish/foot) |
| Genre / Domain | Angle + Grade conditioning |

This notebook tokenizes climbing routes from **both** supported boards:

- Tension Board 2 Mirror
- Kilter Board Original

The board-specific details are stored in `configs/tb2.json` and `configs/kilter.json`.
This version defines the tokenization helpers inline as the notebook needs them.



In [ ]:
from __future__ import annotations

import ast
import json
import random
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

### Board configuration helpers

In [ ]:
# Find the project root and load board configuration JSON files.
def find_project_root(start: str | Path | None = None) -> Path:
    """Walk upward until the repository root markers are found.

    The project root is identified by both ``pyproject.toml`` and ``configs``.
    If neither marker pair is found, the resolved starting directory is returned
    so callers still have a deterministic base path.
    """
    current = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").exists():
            return candidate
    return current

@dataclass(frozen=True)
class BoardConfig:
    """Configuration for a single climbing board.
    
    This dataclass stores all board-specific settings needed for
    data loading, tokenization, and model training.
    
    Attributes:
        board_key: Short identifier (e.g., "tb2", "kilter")
        display_name: Human-readable name (e.g., "Tension Board 2 Mirror")
        token_prefix: Namespace for hold tokens (e.g., "TB2", "KILTER")
        db_path: Path to the SQLite database
        layout_id: Which layout in the database to use
        max_angle: Filter out routes steeper than this (None = no filter)
        min_fa_date: Filter out routes first ascended before this date
        placement_y_max: Filter out placements above this Y coordinate
        include_mirror_placement_id: Whether to include mirror info (TB2 only)
        role_definitions: Maps semantic role names to numeric IDs
        boardlib_database_command: Command to download the database
        boardlib_images_command: Command to download board images
        notes: Additional notes about the configuration
    """
    board_key: str
    display_name: str
    token_prefix: str
    db_path: Path
    layout_id: int
    max_angle: float | None
    min_fa_date: str | None
    placement_y_max: float | None
    include_mirror_placement_id: bool
    role_definitions: dict[str, int]
    boardlib_database_command: str | None = None
    boardlib_images_command: str | None = None
    notes: tuple[str, ...] = ()

    @property
    def role_id_to_name(self) -> dict[int, str]:
        """Reverse mapping from numeric role IDs to semantic role names.
        
        Example: {5: 'start', 6: 'middle', 7: 'finish', 8: 'foot'} for TB2
        """
        return {int(role_id): name for name, role_id in self.role_definitions.items()}

    @property
    def board_token(self) -> str:
        """The special token representing this board.
        
        Example: "<BOARD_TB2>" or "<BOARD_KILTER>"
        """
        return f"<BOARD_{self.token_prefix}>"

    def resolve_db_path(self, project_root: Path | None = None) -> Path:
        """Resolve the database path relative to the project root.
        
        If db_path is absolute, return it as-is.
        Otherwise, resolve it relative to the project root.
        """
        project_root = project_root or find_project_root()
        return self.db_path if self.db_path.is_absolute() else project_root / self.db_path

def load_board_config(board_key: str, config_dir: str | Path | None = None) -> BoardConfig:
    """Load a single board configuration from a JSON file.
    
    Args:
        board_key: Board identifier (e.g., "tb2", "kilter")
        config_dir: Directory containing config JSON files
        
    Returns:
        BoardConfig dataclass with all board settings
        
    Raises:
        FileNotFoundError: If the config file doesn't exist
    """
    project_root = find_project_root()
    config_dir = Path(config_dir) if config_dir is not None else project_root / "configs"
    path = config_dir / f"{board_key}.json"
    if not path.exists():
        available = sorted(p.stem for p in config_dir.glob("*.json"))
        raise FileNotFoundError(
            f"Unknown board config '{board_key}'. Available: {available}"
        )

    payload = json.loads(path.read_text(encoding="utf-8"))
    return BoardConfig(
        board_key=str(payload["board_key"]),
        display_name=str(payload["display_name"]),
        token_prefix=str(payload["token_prefix"]),
        db_path=Path(payload["db_path"]),
        layout_id=int(payload["layout_id"]),
        max_angle=None if payload.get("max_angle") is None else float(payload["max_angle"]),
        min_fa_date=payload.get("min_fa_date"),
        placement_y_max=None if payload.get("placement_y_max") is None else float(payload["placement_y_max"]),
        include_mirror_placement_id=bool(payload.get("include_mirror_placement_id", False)),
        role_definitions={str(k): int(v) for k, v in payload["role_definitions"].items()},
        boardlib_database_command=payload.get("boardlib_database_command"),
        boardlib_images_command=payload.get("boardlib_images_command"),
        notes=tuple(payload.get("notes", [])),
    )

def load_board_configs(board_keys: list[str] | tuple[str, ...]) -> list[BoardConfig]:
    """Load multiple board configurations.
    
    Args:
        board_keys: List of board identifiers
        
    Returns:
        List of BoardConfig dataclasses
    """
    return [load_board_config(board_key) for board_key in board_keys]

## Load board configurations

Each board has its own configuration file (`configs/tb2.json`, `configs/kilter.json`) that specifies:

- **`layout_id`**: Which board layout to use (TB2 Mirror = 10, Kilter Original = 1)
- **`role_definitions`**: Maps semantic role names to board-specific role IDs
  - TB2: start=5, middle=6, finish=7, foot=8
  - Kilter: start=12, middle=13, finish=14, foot=15
- **`max_angle`**: We filter out climbs at extreme angles (>50° for TB2, >55° for Kilter) because those grades are biased toward elite climbers
- **`token_prefix`**: The namespace prefix for hold tokens ("TB2" vs "KILTER")
- **`include_mirror_placement_id`**: Whether to include mirror information (TB2 has symmetric left/right holds)

This configuration-driven approach means we can add new boards by creating a new JSON config file, without changing any code.



In [ ]:
configs = load_board_configs(["tb2", "kilter"])
configs

### Database loading helpers

In [ ]:
# Query each BoardLib SQLite database and attach board identity columns.
def build_climbs_query(config: BoardConfig) -> tuple[str, list]:
    """Build a SQL query for climbs data with board-specific filters.
    
    The query joins climbs, layouts, products, climb_stats, and difficulty_grades
    tables, applying filters for:
    - layout_id: Which board layout to use
    - max_angle: Exclude routes steeper than this
    - min_fa_date: Exclude routes first ascended before this date
    - display_difficulty IS NOT NULL: Only routes with difficulty ratings
    - is_listed = 1: Only publicly listed routes
    
    Args:
        config: Board configuration
        
    Returns:
        Tuple of (SQL query string, list of query parameters)
    """
    conditions = [
        "cs.display_difficulty IS NOT NULL",
        "c.is_listed = 1",
        "c.layout_id = ?",
    ]
    params: list = [config.layout_id]

    if config.max_angle is not None:
        conditions.append("cs.angle <= ?")
        params.append(config.max_angle)

    if config.min_fa_date is not None:
        conditions.append("cs.fa_at > ?")
        params.append(config.min_fa_date)

    query = f"""
    SELECT
        c.uuid,
        c.name AS climb_name,
        c.setter_username,
        c.layout_id AS layout_id,
        c.description,
        c.is_nomatch,
        c.is_listed,
        l.name AS layout_name,
        p.name AS board_name,
        c.frames,
        cs.angle,
        cs.display_difficulty,
        dg.boulder_name AS boulder_grade,
        cs.ascensionist_count,
        cs.quality_average,
        cs.fa_at
    FROM climbs c
    JOIN layouts l ON c.layout_id = l.id
    JOIN products p ON l.product_id = p.id
    JOIN climb_stats cs ON c.uuid = cs.climb_uuid
    JOIN difficulty_grades dg ON ROUND(cs.display_difficulty) = dg.difficulty
    WHERE {' AND '.join(conditions)}
    """
    return query, params

def build_placements_query(config: BoardConfig) -> tuple[str, list]:
    """Build a SQL query for placement data with board-specific filters.
    
    The query retrieves hold positions, default roles, material types,
    and (optionally) mirror placement IDs for symmetric holds.
    
    Args:
        config: Board configuration
        
    Returns:
        Tuple of (SQL query string, list of query parameters)
    """
    params: list = [config.layout_id]
    y_condition = ""
    if config.placement_y_max is not None:
        y_condition = " AND h.y <= ?"
        params.append(config.placement_y_max)

    if config.include_mirror_placement_id:
        # TB2 has mirrored holds — include the mirror placement ID
        query = f"""
        SELECT
            p.id AS placement_id,
            h.x,
            h.y,
            p.default_placement_role_id AS default_role_id,
            p.set_id AS set_id,
            s.name AS set_name,
            p_mirror.id AS mirror_placement_id
        FROM placements p
        JOIN holes h ON p.hole_id = h.id
        JOIN sets s ON p.set_id = s.id
        LEFT JOIN holes h_mirror ON h.mirrored_hole_id = h_mirror.id
        LEFT JOIN placements p_mirror
            ON p_mirror.hole_id = h_mirror.id
           AND p_mirror.layout_id = p.layout_id
        WHERE p.layout_id = ?{y_condition}
        """
    else:
        # Kilter doesn't have mirrored holds
        query = f"""
        SELECT
            p.id AS placement_id,
            h.x,
            h.y,
            p.default_placement_role_id AS default_role_id,
            p.set_id AS set_id,
            s.name AS set_name,
            NULL AS mirror_placement_id
        FROM placements p
        JOIN holes h ON p.hole_id = h.id
        JOIN sets s ON p.set_id = s.id
        WHERE p.layout_id = ?{y_condition}
        """
    return query, params

def load_board_data(
    config: BoardConfig,
    project_root: str | Path | None = None,
    max_climbs: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load climbs and placements data for a single board.
    
    Args:
        config: Board configuration
        project_root: Path to project root (for resolving db_path)
        max_climbs: Optional row limit for fast smoke-test loads.
        
    Returns:
        Tuple of (climbs DataFrame, placements DataFrame)
    """
    project_root = Path(project_root) if project_root is not None else find_project_root()
    db_path = config.resolve_db_path(project_root)
    if not db_path.exists():
        raise FileNotFoundError(
            f"Could not find database for board '{config.board_key}': {db_path}"
        )

    climbs_query, climbs_params = build_climbs_query(config)
    placements_query, placements_params = build_placements_query(config)
    if max_climbs is not None:
        if max_climbs < 1:
            raise ValueError("max_climbs must be at least 1.")
        climbs_query = f"{climbs_query}\nORDER BY c.uuid, cs.angle\nLIMIT ?"
        climbs_params = [*climbs_params, int(max_climbs)]

    with sqlite3.connect(db_path) as conn:
        df_climbs = pd.read_sql_query(climbs_query, conn, params=climbs_params)
        df_placements = pd.read_sql_query(placements_query, conn, params=placements_params)

    # Add board identifiers for multi-board processing
    df_climbs["board_key"] = config.board_key
    df_climbs["board_token_prefix"] = config.token_prefix
    df_climbs["board_display_name"] = config.display_name

    df_placements["board_key"] = config.board_key
    df_placements["board_token_prefix"] = config.token_prefix
    df_placements["board_display_name"] = config.display_name

    return df_climbs, df_placements

def load_multi_board_data(
    configs: list[BoardConfig],
    project_root: str | Path | None = None,
    max_climbs_per_board: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and concatenate data from multiple boards.
    
    This function loads data from each board's database and concatenates
    them into unified DataFrames. Board identifiers are preserved in
    the board_key column.
    
    Args:
        configs: List of board configurations
        project_root: Path to project root
        max_climbs_per_board: Optional row limit per board for smoke tests.
        
    Returns:
        Tuple of (combined climbs DataFrame, combined placements DataFrame)
    """
    climb_frames = []
    placement_frames = []

    for config in configs:
        climbs, placements = load_board_data(
            config,
            project_root=project_root,
            max_climbs=max_climbs_per_board,
        )
        climb_frames.append(climbs)
        placement_frames.append(placements)

    return (
        pd.concat(climb_frames, ignore_index=True),
        pd.concat(placement_frames, ignore_index=True),
    )

## Load raw climbs and placement metadata

The data loading step reads from SQLite databases downloaded using BoardLib:

```bash
boardlib database tension data/raw/tb2.db
boardlib database kilter data/raw/kilter.db
```

### What we're loading

**Climbs data** (`df_climbs`): Each row is a climb-angle observation. Key columns:
- `uuid`: Unique climb identifier
- `frames`: The raw string encoding holds and roles, e.g., `p344r5p369r6p603r7`
- `angle`: Wall angle in degrees
- `display_difficulty`: Numeric difficulty score (maps to V-grades)
- `boulder_grade`: Human-readable grade like "6b/V4"

**Placements data** (`df_placements`): Each row is a physical hold position on the board. Key columns:
- `placement_id`: The hold's unique ID within its board
- `x`, `y`: Physical coordinates on the board (in inches)
- `default_role_id`: What role this hold typically plays (hand vs foot)
- `set_name`: Material type ("Wood" or "Plastic")
- `mirror_placement_id`: For TB2, the ID of the symmetric hold on the other side



In [ ]:
df_climbs, df_placements = load_multi_board_data(configs, project_root=ROOT)
print(f"Total climbs loaded: {len(df_climbs):,}")
print(f"Total placements loaded: {len(df_placements):,}")
print()
print("Climbs per board:")
print(df_climbs.groupby("board_key").size())

### Grade and route-tokenization helpers

In [ ]:
# Map BoardLib display difficulties into grouped V-grade tokens.
GRADE_TO_V = {
    10: 0, 11: 0, 12: 0,
    13: 1, 14: 1,
    15: 2,
    16: 3, 17: 3,
    18: 4, 19: 4,
    20: 5, 21: 5,
    22: 6,
    23: 7,
    24: 8, 25: 8,
    26: 9,
    27: 10,
    28: 11,
    29: 12,
    30: 13,
    31: 14,
    32: 15,
    33: 16,
}

def to_grouped_v(display_difficulty: float) -> int:
    """Map a continuous display difficulty to the nearest grouped V grade."""
    rounded = int(round(float(display_difficulty)))
    rounded = max(min(rounded, max(GRADE_TO_V)), min(GRADE_TO_V))
    return GRADE_TO_V[rounded]

def grade_token(display_difficulty: float) -> str:
    """Return the grade-conditioning token for a display difficulty value."""
    return f"<GRADE_V{to_grouped_v(display_difficulty)}>"

# Parse frames, canonicalize holds, and build route-level token sequences.
SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
    "<CLS>",
    "<MASK>",
]

ANGLE_TOKEN_PATTERN = re.compile(r"^<ANGLE_(-?\d+)>$")

GRADE_TOKEN_PATTERN = re.compile(r"^<GRADE_V(\d+)>$")

BOARD_TOKEN_PATTERN = re.compile(r"^<BOARD_([A-Z0-9_]+)>$")

HOLD_TOKEN_PATTERN = re.compile(r"^<([A-Z0-9_]+)_p(\d+)_(start|middle|finish|foot|unknown)>$")

ROLE_SORT_ORDER = {
    "start": 0,
    "middle": 1,
    "foot": 2,
    "finish": 3,
    "unknown": 9,
}

def parse_frames(frames_str: str | None) -> list[tuple[int, int]]:
    """Parse a frames string into ``(placement_id, role_id)`` pairs.

    Frames strings are compact concatenations such as ``p344r5p369r6``. Invalid
    or missing input returns an empty list so callers can skip unusable climbs
    without special-case exception handling.
    """
    if not isinstance(frames_str, str):
        return []
    matches = re.findall(r"p(\d+)r(\d+)", frames_str)
    return [(int(placement_id), int(role_id)) for placement_id, role_id in matches]

def make_placement_lookup(df_placements: pd.DataFrame) -> dict[tuple[str, int], dict]:
    """Build a coordinate/metadata lookup keyed by ``(board_key, placement_id)``."""
    rows = {}
    for _, row in df_placements.iterrows():
        key = (str(row["board_key"]), int(row["placement_id"]))
        rows[key] = row.to_dict()
    return rows

def role_name(role_id: int, config: BoardConfig) -> str:
    """Map a board-specific numeric role ID to a shared semantic role name."""
    return config.role_id_to_name.get(int(role_id), "unknown")

def placement_xy(
    board_key: str,
    placement_id: int,
    placement_lookup: dict[tuple[str, int], dict],
) -> tuple[float, float]:
    """Return raw board coordinates for a placement, or NaNs if unknown."""
    row = placement_lookup.get((str(board_key), int(placement_id)))
    if row is None:
        return (float("nan"), float("nan"))
    return (float(row["x"]), float(row["y"]))

def canonicalize_holds(
    holds: Iterable[tuple[int, int]],
    config: BoardConfig,
    placement_lookup: dict[tuple[str, int], dict],
) -> list[tuple[int, int]]:
    """Sort holds into the canonical route order used by all model inputs.

    Frames preserve setter/storage order, which is not always stable
    across routes or boards. Canonical ordering keeps starts first, hand/foot
    holds in a bottom-to-top scan, and finishes last, giving the models a more
    consistent sequence grammar.
    """
    def key(pair: tuple[int, int]):
        """Sort by semantic role, then board position, then placement ID."""
        placement_id, role_id = pair
        x, y = placement_xy(config.board_key, placement_id, placement_lookup)
        name = role_name(role_id, config)
        return (
            ROLE_SORT_ORDER.get(name, 9),
            y if not np.isnan(y) else 9999.0,
            x if not np.isnan(x) else 9999.0,
            placement_id,
        )

    return sorted(list(holds), key=key)

def board_token(config: BoardConfig) -> str:
    """Return the special conditioning token for a board config."""
    return f"<BOARD_{config.token_prefix}>"

def angle_token(angle: float) -> str:
    """Round a wall angle into the shared angle-token format."""
    return f"<ANGLE_{int(round(float(angle)))}>"

def hold_token(
    placement_id: int,
    role_id: int,
    config: BoardConfig,
) -> str:
    """Return a board-namespaced hold token for a placement and role."""
    semantic_role = role_name(role_id, config)
    return f"<{config.token_prefix}_p{int(placement_id)}_{semantic_role}>"

def tokenize_route(
    row,
    config: BoardConfig,
    placement_lookup: dict[tuple[str, int], dict],
    include_grade: bool = True,
    canonical: bool = True,
) -> list[str]:
    """Tokenize one climb row into the sequence consumed by the models.

    ``include_grade=True`` is used for GPT-style generation, where the target
    grade is a conditioning token. ``include_grade=False`` is used for grade
    prediction so the model cannot read the answer from its input.
    """
    holds = parse_frames(row["frames"])
    if canonical:
        holds = canonicalize_holds(holds, config, placement_lookup)

    tokens = [
        "<BOS>",
        board_token(config),
        angle_token(row["angle"]),
    ]
    if include_grade:
        tokens.append(grade_token(row["display_difficulty"]))

    tokens.extend(hold_token(placement_id, role_id, config) for placement_id, role_id in holds)
    tokens.append("<EOS>")
    return tokens

def build_route_records(
    df_climbs: pd.DataFrame,
    configs_by_key: dict[str, BoardConfig],
    placement_lookup: dict[tuple[str, int], dict],
) -> pd.DataFrame:
    """Create one training/evaluation record per climb-angle row.

    The returned frame keeps both human-readable route metadata and model-ready
    token sequences, which lets downstream scripts save compact CSV summaries
    while still retaining the richer JSONL training artifacts.
    """
    records: list[dict] = []

    for _, row in df_climbs.iterrows():
        board_key = str(row["board_key"])
        config = configs_by_key[board_key]
        holds = canonicalize_holds(parse_frames(row["frames"]), config, placement_lookup)
        if not holds:
            continue

        hold_tokens = [hold_token(p, r, config) for p, r in holds]
        semantic_roles = [role_name(r, config) for _, r in holds]

        tokens_with_grade = tokenize_route(
            row,
            config=config,
            placement_lookup=placement_lookup,
            include_grade=True,
            canonical=True,
        )
        tokens_no_grade = tokenize_route(
            row,
            config=config,
            placement_lookup=placement_lookup,
            include_grade=False,
            canonical=True,
        )

        records.append(
            {
                "uuid": row["uuid"],
                "board_key": board_key,
                "board_display_name": row["board_display_name"],
                "board_token_prefix": row["board_token_prefix"],
                "board_token": board_token(config),
                "climb_name": row["climb_name"],
                "setter_username": row.get("setter_username"),
                "layout_id": int(row["layout_id"]),
                "layout_name": row.get("layout_name"),
                "board_name": row.get("board_name"),
                "frames": row["frames"],
                "angle": float(row["angle"]),
                "display_difficulty": float(row["display_difficulty"]),
                "grouped_v": int(to_grouped_v(row["display_difficulty"])),
                "boulder_grade": row.get("boulder_grade"),
                "ascensionist_count": row.get("ascensionist_count"),
                "quality_average": row.get("quality_average"),
                "fa_at": row.get("fa_at"),
                "n_holds": len(holds),
                "n_start": semantic_roles.count("start"),
                "n_middle": semantic_roles.count("middle"),
                "n_foot": semantic_roles.count("foot"),
                "n_finish": semantic_roles.count("finish"),
                "holds": holds,
                "hold_tokens": hold_tokens,
                "tokens_with_grade": tokens_with_grade,
                "tokens_no_grade": tokens_no_grade,
                "sequence_with_grade": " ".join(tokens_with_grade),
                "sequence_no_grade": " ".join(tokens_no_grade),
            }
        )

    return pd.DataFrame(records)

## Build unified route records

This is the core tokenization step. For each climb, we:

1. **Parse the frames string**: Convert `p344r5p369r6p603r7` into a list of `(placement_id, role_id)` tuples

2. **Map role IDs to semantic roles**: Convert board-specific role IDs (5→start, 6→middle, etc.) to shared semantic names

3. **Canonicalize hold order**: Sort holds by (role priority, y-position, x-position). This is important because:
   - The same climb can be represented with holds in any order in the database
   - Transformers need consistent input ordering to learn patterns
   - This is analogous to how NLP tokenizers normalize text (lowercasing, etc.)

4. **Generate token sequences**: Create two versions of each route:
   - `sequence_with_grade`: `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> ... <EOS>`
   - `sequence_no_grade`: `<BOS> <BOARD_TB2> <ANGLE_40> <TB2_p344_start> ... <EOS>` (grade removed)

The grade-included version is used for the GPT generator (which predicts the next token, including grade). The grade-excluded version is used for the grade predictor (which receives the route without knowing the grade and must predict it).



In [ ]:
configs_by_key = {config.board_key: config for config in configs}
configs_by_prefix = {config.token_prefix: config for config in configs}
placement_lookup = make_placement_lookup(df_placements)

df_routes = build_route_records(
    df_climbs=df_climbs,
    configs_by_key=configs_by_key,
    placement_lookup=placement_lookup,
)
print(f"Tokenized routes: {len(df_routes):,}")
print()
df_routes[["board_key", "angle", "display_difficulty", "sequence_with_grade"]].head()

## Example tokenized routes

Let's look at what the tokenized routes actually look like. This is the "text" that our transformer models will read.



In [ ]:
for _, row in df_routes.groupby("board_key").head(2).iterrows():
    print(f"Board: {row['board_key']}")
    print(f"  Angle: {row['angle']}°")
    print(f"  Grade: {row['boulder_grade']} (V{row['grouped_v']})")
    print(f"  Tokens: {row['sequence_with_grade']}")
    print()

### Vocabulary helpers

In [ ]:
# Build the shared vocabulary and encode/decode token strings.
def build_vocab(df_routes: pd.DataFrame) -> tuple[list[str], dict[str, int], dict[int, str]]:
    """Build the shared token vocabulary from grade-conditioned sequences."""
    all_tokens: list[str] = []
    for tokens in df_routes["tokens_with_grade"]:
        all_tokens.extend(tokens)

    vocab_tokens = list(SPECIAL_TOKENS)
    for token in sorted(set(all_tokens)):
        if token not in vocab_tokens:
            vocab_tokens.append(token)

    stoi = {token: idx for idx, token in enumerate(vocab_tokens)}
    itos = {idx: token for token, idx in stoi.items()}
    return vocab_tokens, stoi, itos

def encode(tokens: Iterable[str], stoi: dict[str, int]) -> list[int]:
    """Convert tokens to integer IDs, using ``<UNK>`` for unseen tokens."""
    unk_id = stoi["<UNK>"]
    return [stoi.get(token, unk_id) for token in tokens]

def decode(ids: Iterable[int], itos: dict[int, str]) -> list[str]:
    """Convert integer IDs back to token strings."""
    return [itos.get(int(idx), "<UNK>") for idx in ids]

## Build the shared vocabulary

### What is a vocabulary?

In NLP, the **vocabulary** (or "vocab") is the set of all possible tokens the model can produce or understand. For GPT-3, this is ~50,000 BPE tokens. For our climbing model, it includes:

1. **Special tokens** (6): `<PAD>`, `<UNK>`, `<BOS>`, `<EOS>`, `<CLS>`, `<MASK>`
2. **Board tokens** (2): `<BOARD_TB2>`, `<BOARD_KILTER>`
3. **Angle tokens** (~6): `<ANGLE_30>`, `<ANGLE_35>`, `<ANGLE_40>`, etc.
4. **Grade tokens** (~17): `<GRADE_V0>` through `<GRADE_V16>`
5. **Hold tokens** (~1000+): One per placement per board per role

### Why board-namespaced hold tokens?

Placement ID 344 on TB2 refers to a completely different physical hold than placement ID 344 on Kilter (the latter doesn't exist). By prefixing with the board name (`TB2_p344_start` vs `KILTER_p344_start`), we ensure the model treats these as distinct tokens.

This is analogous to how multilingual LLMs use language-specific subword tokens — the same byte sequence can mean different things in different languages.

### String-to-integer mapping

Transformers operate on integer indices, not strings. The `stoi` (string-to-integer) and `itos` (integer-to-string) dictionaries provide this mapping, similar to how HuggingFace tokenizers work.



In [ ]:
vocab_tokens, stoi, itos = build_vocab(df_routes)

print(f"Vocabulary size: {len(stoi):,}")
print()
print("First 20 tokens (special + board tokens):")
print(vocab_tokens[:20])
print()
hold_tokens = [t for t in vocab_tokens if t.startswith('<') and '_p' in t]
angle_tokens = [t for t in vocab_tokens if t.startswith('<ANGLE_')]
grade_tokens = [t for t in vocab_tokens if t.startswith('<GRADE_')]
board_tokens = [t for t in vocab_tokens if t.startswith('<BOARD_')]
special_tokens = [t for t in vocab_tokens if t in ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<CLS>', '<MASK>']]

print(f"Special tokens: {len(special_tokens)}")
print(f"Board tokens: {len(board_tokens)}")
print(f"Angle tokens: {len(angle_tokens)}")
print(f"Grade tokens: {len(grade_tokens)}")
print(f"Hold tokens: {len(hold_tokens)}")

### Split helpers

In [ ]:
# Assign train/validation/test splits at the logical-climb group level.
def safe_train_test_split(
    df: pd.DataFrame,
    test_size: float,
    random_state: int,
    stratify_col: str | None = None,
):
    """Split a DataFrame with optional stratification and graceful fallback.

    scikit-learn raises when a requested stratum is too small. The tokenization
    pipeline prefers stratified splits when possible, but falls back to an
    unstratified split rather than failing on tiny smoke-test subsets.
    """
    stratify = None
    if stratify_col is not None and stratify_col in df.columns:
        counts = df[stratify_col].value_counts()
        if len(counts) > 1 and counts.min() >= 2:
            stratify = df[stratify_col]

    try:
        return train_test_split(
            df,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )
    except ValueError:
        return train_test_split(
            df,
            test_size=test_size,
            random_state=random_state,
            stratify=None,
        )

def assign_group_splits(
    df: pd.DataFrame,
    group_cols: list[str],
    test_size: float,
    val_size_within_temp: float,
    random_state: int,
    stratify_col: str | None = None,
) -> pd.Series:
    """Assign train/val/test splits at group level.

    This prevents multiple rows for the same logical climb, for example the
    same UUID at several angles, from being distributed across different
    splits. The returned Series is indexed like ``df`` and contains
    ``train``, ``val``, or ``test``.
    """
    group_df = df[group_cols + ([stratify_col] if stratify_col else [])].copy()
    group_df = group_df.drop_duplicates(group_cols).reset_index(drop=True)

    train_groups, temp_groups = safe_train_test_split(
        group_df,
        test_size=test_size,
        random_state=random_state,
        stratify_col=stratify_col,
    )
    val_groups, test_groups = safe_train_test_split(
        temp_groups,
        test_size=val_size_within_temp,
        random_state=random_state,
        stratify_col=stratify_col,
    )

    def key_frame(frame: pd.DataFrame) -> set[tuple]:
        """Return stringified group keys so pandas dtypes cannot affect joins."""
        return set(map(tuple, frame[group_cols].astype(str).values.tolist()))

    train_keys = key_frame(train_groups)
    val_keys = key_frame(val_groups)
    test_keys = key_frame(test_groups)

    def split_for_row(row) -> str:
        """Map one original row back to its group-level split assignment."""
        key = tuple(str(row[col]) for col in group_cols)
        if key in train_keys:
            return "train"
        if key in val_keys:
            return "val"
        if key in test_keys:
            return "test"
        raise KeyError(f"Could not assign split for group key {key}")

    return df.apply(split_for_row, axis=1)

## Train/validation/test splits

### Why stratified splitting?

We split data into train (80%), validation (10%), and test (10%) sets. Crucially, we **stratify by `board_key × grouped_v`** — this ensures that:

1. Both boards (TB2 and Kilter) are represented in all splits
2. All difficulty levels (V0 through V16) are represented in all splits

Without stratification, we might end up with all V14 climbs in the test set and none in training, which would make evaluation meaningless.

This is the same principle as stratified splitting in NLP, where you ensure all languages or domains are represented in each split.



In [ ]:
df_routes["ids_with_grade"] = df_routes["tokens_with_grade"].apply(lambda tokens: encode(tokens, stoi))
df_routes["ids_no_grade"] = df_routes["tokens_no_grade"].apply(lambda tokens: encode(tokens, stoi))
df_routes["split_stratum"] = df_routes["board_key"].astype(str) + "__V" + df_routes["grouped_v"].astype(str)
df_routes["split"] = assign_group_splits(
    df_routes,
    group_cols=["board_key", "uuid"],
    test_size=0.20,
    val_size_within_temp=0.50,
    random_state=3,
    stratify_col="split_stratum",
)

df_routes.groupby(["board_key", "split"]).size().unstack(fill_value=0)

### Token metadata helpers

In [ ]:
# Attach board, role, placement, and coordinate metadata to each token.
def build_token_metadata(
    vocab_tokens: list[str],
    stoi: dict[str, int],
    df_placements: pd.DataFrame,
    placement_lookup: dict[tuple[str, int], dict],
    configs_by_prefix: dict[str, BoardConfig],
) -> pd.DataFrame:
    """Build per-token metadata used for coordinate features and plotting.

    Hold tokens receive raw coordinates, normalized coordinates in ``[-1, 1]``,
    role labels, and board identity. Non-hold tokens keep neutral coordinate
    features so the grade predictor can safely index every token ID.
    """
    bounds = {}
    for board_key, frame in df_placements.groupby("board_key"):
        xs = frame["x"].astype(float)
        ys = frame["y"].astype(float)
        bounds[str(board_key)] = {
            "x_min": float(xs.min()),
            "x_max": float(xs.max()),
            "y_min": float(ys.min()),
            "y_max": float(ys.max()),
        }

    def normalize(value: float, lo: float, hi: float) -> float:
        """Scale one coordinate into ``[-1, 1]`` with safe missing-value handling."""
        if pd.isna(value) or hi == lo:
            return 0.0
        return 2 * ((float(value) - lo) / (hi - lo)) - 1

    rows: list[dict] = []

    for token in vocab_tokens:
        meta = {
            "token": token,
            "token_id": stoi[token],
            "kind": "special",
            "board_key": None,
            "board_token_prefix": None,
            "placement_id": np.nan,
            "role": None,
            "x": np.nan,
            "y": np.nan,
            "x_norm": 0.0,
            "y_norm": 0.0,
            "is_hold": 0,
            "angle": np.nan,
            "grouped_v": np.nan,
        }

        hold_match = HOLD_TOKEN_PATTERN.match(token)
        if hold_match:
            prefix = hold_match.group(1)
            placement_id = int(hold_match.group(2))
            role = hold_match.group(3)
            config = configs_by_prefix[prefix]
            board_key = config.board_key
            row = placement_lookup.get((board_key, placement_id), {})
            x = float(row.get("x", np.nan))
            y = float(row.get("y", np.nan))
            board_bounds = bounds.get(board_key, {"x_min": 0, "x_max": 1, "y_min": 0, "y_max": 1})

            meta.update(
                {
                    "kind": "hold",
                    "board_key": board_key,
                    "board_token_prefix": prefix,
                    "placement_id": placement_id,
                    "role": role,
                    "x": x,
                    "y": y,
                    "x_norm": normalize(x, board_bounds["x_min"], board_bounds["x_max"]),
                    "y_norm": normalize(y, board_bounds["y_min"], board_bounds["y_max"]),
                    "is_hold": 1,
                }
            )

        angle_match = ANGLE_TOKEN_PATTERN.match(token)
        if angle_match:
            meta.update({"kind": "angle", "angle": int(angle_match.group(1))})

        grade_match = GRADE_TOKEN_PATTERN.match(token)
        if grade_match:
            meta.update({"kind": "grade", "grouped_v": int(grade_match.group(1))})

        board_match = BOARD_TOKEN_PATTERN.match(token)
        if board_match:
            prefix = board_match.group(1)
            config = configs_by_prefix.get(prefix)
            meta.update(
                {
                    "kind": "board",
                    "board_key": None if config is None else config.board_key,
                    "board_token_prefix": prefix,
                }
            )

        rows.append(meta)

    return pd.DataFrame(rows)

def vocab_payload(
    stoi: dict[str, int],
    itos: dict[int, str],
    configs_by_key: dict[str, BoardConfig],
) -> dict:
    """Package vocabulary and board metadata for JSON serialization."""
    return {
        "stoi": stoi,
        "itos": {str(k): v for k, v in itos.items()},
        "special_tokens": SPECIAL_TOKENS,
        "boards": {
            board_key: {
                "token_prefix": config.token_prefix,
                "board_token": board_token(config),
                "role_definitions": config.role_definitions,
            }
            for board_key, config in configs_by_key.items()
        },
        "grade_to_v": {str(k): v for k, v in GRADE_TO_V.items()},
    }

## Token metadata

### Why metadata matters

Each hold token carries rich metadata that the model can use:

- **Physical coordinates** (`x`, `y`): Where the hold is on the board
- **Normalized coordinates** (`x_norm`, `y_norm`): Scaled to [-1, 1] per board, so the model doesn't need to learn different coordinate scales
- **Semantic role** (`start`, `middle`, `finish`, `foot`): What the hold is used for
- **Board identity** (`board_key`): Which board this hold belongs to

The grade predictor uses these coordinate features as additional embeddings alongside the token embeddings. This is similar to how some LLMs inject positional embeddings or segment embeddings — we're giving the model extra structured information about each token.



In [ ]:
df_token_meta = build_token_metadata(
    vocab_tokens=vocab_tokens,
    stoi=stoi,
    df_placements=df_placements,
    placement_lookup=placement_lookup,
    configs_by_prefix=configs_by_prefix,
)

print("Token metadata columns:")
print(df_token_meta.columns.tolist())
print()
print("Example hold token metadata:")
df_token_meta[df_token_meta["kind"] == "hold"].head()

### JSON output helpers

In [ ]:
# Write JSON artifacts after converting NumPy/pandas values to plain Python values.
def json_safe(obj: Any) -> Any:
    """Convert NumPy/pandas values into JSON-serializable Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)
    if isinstance(obj, np.ndarray):
        return json_safe(obj.tolist())
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def write_json(path: str | Path, payload: Any) -> None:
    """Write an object as indented UTF-8 JSON after ``json_safe`` cleanup."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2), encoding="utf-8")

## Save artifacts

We save several files that will be consumed by later notebooks:

1. **`route_sequences.csv`**: The main tokenized dataset with train/val/test splits
2. **`routes_tokenized.jsonl`**: Same data in JSON Lines format (one JSON object per route)
3. **`token_vocab.json`**: The vocabulary mapping (stoi and itos)
4. **`token_metadata.csv`**: Metadata for each token (coordinates, roles, etc.)
5. **`placement_metadata.csv`**: Physical placement information
6. **`board_summary.csv`**: Aggregate statistics per board



In [ ]:
OUT = ROOT / "data" / "processed" / "tokenized"
OUT.mkdir(parents=True, exist_ok=True)

csv_cols = [
    "uuid", "board_key", "board_display_name", "board_token_prefix", "board_token",
    "climb_name", "setter_username", "layout_id", "layout_name", "board_name",
    "frames", "angle", "display_difficulty", "grouped_v", "boulder_grade",
    "ascensionist_count", "quality_average", "fa_at",
    "n_holds", "n_start", "n_middle", "n_foot", "n_finish",
    "sequence_with_grade", "sequence_no_grade", "split",
]
df_routes[csv_cols].to_csv(OUT / "route_sequences.csv", index=False)

df_placements.to_csv(OUT / "placement_metadata.csv", index=False)

df_token_meta.to_csv(OUT / "token_metadata.csv", index=False)

write_json(OUT / "token_vocab.json", vocab_payload(stoi, itos, configs_by_key))

with (OUT / "routes_tokenized.jsonl").open("w", encoding="utf-8") as handle:
    for record in df_routes.to_dict(orient="records"):
        handle.write(json.dumps(json_safe(record)) + "\n")

board_summary = (
    df_routes.groupby("board_key")
    .agg(
        n_routes=("uuid", "count"),
        mean_angle=("angle", "mean"),
        mean_display_difficulty=("display_difficulty", "mean"),
        mean_holds=("n_holds", "mean"),
    )
    .reset_index()
)
board_summary.to_csv(OUT / "board_summary.csv", index=False)

print("Saved artifacts to:", OUT)
print(f"  - route_sequences.csv ({len(df_routes):,} rows)")
print(f"  - routes_tokenized.jsonl")
print(f"  - token_vocab.json ({len(stoi):,} tokens)")
print(f"  - token_metadata.csv ({len(df_token_meta):,} rows)")
print(f"  - placement_metadata.csv ({len(df_placements):,} rows)")
print(f"  - board_summary.csv")